# 07 - Insights para Dashboard Looker Studio

Gera **views de insights** sobre as tabelas Gold para alimentar o dashboard no Looker Studio.

**Squad 2 — Real Time for Business | Dupla 2**  
**Integrantes:** Kauan & Leandro  
**Origem:** Tabelas Gold em `workspace.default` (KPIs de volume, ticket médio e cancelamento)

### Regras de negócio aplicadas:

| KPI | Insight gerado | Regra |
| --- | --- | --- |
| **Volume (5min)** | Tendência + variação % | Alta se >+10%, Queda se <-10%, Estável otherwise |
| **Ticket médio** | Ranking + impacto | Top 2 = Alto impacto, 3-4 = Médio, demais = Baixo |
| **Cancelamento** | Status de severidade | >10% = Crítico, 5-10% = Atenção, <5% = OK |

### Paleta de cores (padrão visual):

| Status | Cor | Hex |
| --- | --- | --- |
| OK / Sucesso | Verde | `#2E7D32` |
| Alerta | Amarelo | `#F2B705` |
| Erro / Crítico | Vermelho | `#C0392B` |
| Neutro | Cinza | `#8C8C8C` |

> Views criadas: `vw_insight_volume_pedidos`, `vw_insight_ticket_pagamento`, `vw_insight_cancelamento`, `vw_insights_dashboard`

In [0]:
%sql
-- ============================================================================
-- INSIGHTS: Views de análise derivada para o dashboard Looker Studio
-- Segue padrão visual: verde (#2E7D32) = OK, amarelo (#F2B705) = alerta,
-- vermelho (#C0392B) = erro, cinza (#8C8C8C) = neutro
-- ============================================================================

-- Insight KPI 1: Volume de pedidos — tendência e variação percentual vs janela anterior
CREATE OR REPLACE VIEW workspace.default.vw_insight_volume_pedidos AS
WITH base AS (
    SELECT *,
           LAG(qtd_pedidos) OVER (ORDER BY dt_inicio_janela) AS qtd_anterior,
           LAG(valor_total_pedidos) OVER (ORDER BY dt_inicio_janela) AS valor_anterior
    FROM workspace.default.gold_kpi_volume_pedidos_5min
)
SELECT
    dt_inicio_janela,
    dt_fim_janela,
    qtd_pedidos,
    valor_total_pedidos,
    ticket_medio,
    qtd_anterior,
    ROUND(
        CASE WHEN qtd_anterior IS NULL OR qtd_anterior = 0 THEN NULL
             ELSE (qtd_pedidos - qtd_anterior) * 100.0 / qtd_anterior
        END, 2
    ) AS variacao_pct,
    CASE 
        WHEN qtd_anterior IS NULL THEN 'Sem comparação'
        WHEN qtd_pedidos > qtd_anterior * 1.10 THEN 'Alta'
        WHEN qtd_pedidos < qtd_anterior * 0.90 THEN 'Queda'
        ELSE 'Estável'
    END AS tendencia,
    CASE 
        WHEN qtd_anterior IS NULL THEN '#8C8C8C'
        WHEN qtd_pedidos < qtd_anterior * 0.90 THEN '#C0392B'
        WHEN qtd_pedidos < qtd_anterior * 0.95 THEN '#F2B705'
        ELSE '#2E7D32'
    END AS cor_status,
    dt_calculo
FROM base;

-- Insight KPI 2: Ticket médio por pagamento — ranking e classificação de impacto
CREATE OR REPLACE VIEW workspace.default.vw_insight_ticket_pagamento AS
WITH base AS (
    SELECT *,
           ROW_NUMBER() OVER (ORDER BY valor_total DESC) AS ranking,
           SUM(valor_total) OVER () AS valor_total_geral
    FROM workspace.default.gold_kpi_ticket_medio_por_pagamento
)
SELECT
    metodo_pagamento,
    qtd_pedidos,
    valor_total,
    ticket_medio,
    ranking,
    ROUND(valor_total * 100.0 / valor_total_geral, 2) AS pct_participacao,
    CASE 
        WHEN ranking <= 2 THEN 'Alto impacto'
        WHEN ranking <= 4 THEN 'Médio impacto'
        ELSE 'Baixo impacto'
    END AS classe_impacto,
    CASE 
        WHEN ranking <= 2 THEN '#2E7D32'
        WHEN ranking <= 4 THEN '#F2B705'
        ELSE '#8C8C8C'
    END AS cor_status,
    dt_calculo
FROM base;

-- Insight KPI 3: Taxa de cancelamento — classificação de severidade
CREATE OR REPLACE VIEW workspace.default.vw_insight_cancelamento AS
SELECT
    dt_inicio_janela,
    dt_fim_janela,
    total_pedidos,
    pedidos_cancelados,
    taxa_cancelamento,
    CASE 
        WHEN taxa_cancelamento > 10 THEN 'Crítico'
        WHEN taxa_cancelamento > 5 THEN 'Atenção'
        ELSE 'OK'
    END AS status,
    CASE 
        WHEN taxa_cancelamento > 10 THEN '#C0392B'
        WHEN taxa_cancelamento > 5 THEN '#F2B705'
        ELSE '#2E7D32'
    END AS cor_status,
    dt_calculo
FROM workspace.default.gold_kpi_taxa_cancelamento_hora;

-- View consolidada para KPI cards no topo do dashboard
CREATE OR REPLACE VIEW workspace.default.vw_insights_dashboard AS
SELECT
    'KPI 1 — Volume (5min)' AS kpi,
    CAST((SELECT MAX(qtd_pedidos) FROM workspace.default.vw_insight_volume_pedidos
          WHERE dt_inicio_janela = (SELECT MAX(dt_inicio_janela) FROM workspace.default.vw_insight_volume_pedidos)) AS DECIMAL(18,2)) AS valor_destaque,
    'pedidos na última janela' AS descricao,
    (SELECT tendencia FROM workspace.default.vw_insight_volume_pedidos
     WHERE dt_inicio_janela = (SELECT MAX(dt_inicio_janela) FROM workspace.default.vw_insight_volume_pedidos) LIMIT 1) AS classificacao,
    (SELECT cor_status FROM workspace.default.vw_insight_volume_pedidos
     WHERE dt_inicio_janela = (SELECT MAX(dt_inicio_janela) FROM workspace.default.vw_insight_volume_pedidos) LIMIT 1) AS cor
UNION ALL
SELECT
    'KPI 2 — Ticket médio',
    CAST((SELECT MAX(ticket_medio) FROM workspace.default.vw_insight_ticket_pagamento WHERE ranking = 1) AS DECIMAL(18,2)),
    'ticket médio (top pagamento)',
    (SELECT classe_impacto FROM workspace.default.vw_insight_ticket_pagamento WHERE ranking = 1 LIMIT 1),
    (SELECT cor_status FROM workspace.default.vw_insight_ticket_pagamento WHERE ranking = 1 LIMIT 1)
UNION ALL
SELECT
    'KPI 3 — Cancelamento',
    CAST((SELECT MAX(taxa_cancelamento) FROM workspace.default.vw_insight_cancelamento
          WHERE dt_inicio_janela = (SELECT MAX(dt_inicio_janela) FROM workspace.default.vw_insight_cancelamento)) AS DECIMAL(18,2)),
    'taxa na última hora (%)',
    (SELECT status FROM workspace.default.vw_insight_cancelamento
     WHERE dt_inicio_janela = (SELECT MAX(dt_inicio_janela) FROM workspace.default.vw_insight_cancelamento) LIMIT 1),
    (SELECT cor_status FROM workspace.default.vw_insight_cancelamento
     WHERE dt_inicio_janela = (SELECT MAX(dt_inicio_janela) FROM workspace.default.vw_insight_cancelamento) LIMIT 1);

-- Prévia dos insights consolidados
SELECT * FROM workspace.default.vw_insights_dashboard;

In [0]:
%sql
-- Prévia: Insight KPI 1 — Volume com tendência
SELECT dt_inicio_janela, qtd_pedidos, qtd_anterior, variacao_pct, tendencia, cor_status
FROM workspace.default.vw_insight_volume_pedidos
ORDER BY dt_inicio_janela DESC
LIMIT 10;

In [0]:
%sql
-- Prévia: Insight KPI 2 — Ticket médio com ranking e impacto
SELECT metodo_pagamento, qtd_pedidos, valor_total, ticket_medio, ranking, pct_participacao, classe_impacto, cor_status
FROM workspace.default.vw_insight_ticket_pagamento
ORDER BY ranking;

In [0]:
%sql
-- Prévia: Insight KPI 3 — Cancelamento com status
SELECT dt_inicio_janela, total_pedidos, pedidos_cancelados, taxa_cancelamento, status, cor_status
FROM workspace.default.vw_insight_cancelamento
ORDER BY dt_inicio_janela DESC
LIMIT 10;